In [ ]:
"""
Gold 1-Minute Experiments â€” Kaggle Script
=========================================
Runs a grid of experiments to find the best model configuration.
Strategies tested:
  - Different horizons (5, 20, 60)
  - Different targets (raw, vol-filtered, momentum-confirmed, 3-class)
  - Different feature sets (basic, full microstructure)
  - Different model capacities (conservative vs aggressive)

Outputs:
  - experiment_results.csv  (all metrics)
  - best_model.pkl          (winning model + config)
  - best_config.json        (summary of winner)
"""

import os
import sys
import glob
import json
import warnings
import time
from datetime import datetime

import numpy as np
import pandas as pd
import joblib

import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score, roc_auc_score, log_loss,
    precision_score, recall_score, f1_score,
    confusion_matrix, mean_squared_error
)
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings('ignore')
np.random.seed(42)

print("=" * 70)
print("GOLD 1-MINUTE MODEL EXPERIMENTS")
print("=" * 70)
print(f"Start: {datetime.now().isoformat()}")

# =============================================================================
# CONFIG
# =============================================================================
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
FEATURE_TOP_K = 35
EARLY_STOP = 150
MAX_ROUNDS = 3000

# Two LGB parameter sets
LGB_CONSERVATIVE = {
    'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
    'num_leaves': 31, 'max_depth': 5, 'min_data_in_leaf': 300,
    'learning_rate': 0.01, 'feature_fraction': 0.65,
    'bagging_fraction': 0.7, 'bagging_freq': 5,
    'lambda_l1': 0.5, 'lambda_l2': 2.0, 'max_bin': 127,
    'verbose': -1, 'seed': 42, 'deterministic': True,
}

LGB_AGGRESSIVE = {
    'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
    'num_leaves': 127, 'max_depth': 10, 'min_data_in_leaf': 50,
    'learning_rate': 0.03, 'feature_fraction': 0.8,
    'bagging_fraction': 0.8, 'bagging_freq': 3,
    'lambda_l1': 0.05, 'lambda_l2': 0.5, 'max_bin': 255,
    'verbose': -1, 'seed': 42, 'deterministic': True,
}

# =============================================================================
# LOAD DATA
# =============================================================================
print("\n[1/5] Loading data...")
csv_files = glob.glob('/kaggle/input/**/*.csv', recursive=True)
preferred = [f for f in csv_files if 'xau' in f.lower() or 'gold' in f.lower()]
csv_path = preferred[0] if preferred else csv_files[0]
print(f"  File: {csv_path}")

with open(csv_path, 'r') as fh:
    first = fh.readline()
    sep = ';' if ';' in first else (',' if ',' in first else None)

df = pd.read_csv(csv_path, sep=sep, engine='python')
print(f"  Raw shape: {df.shape}")

df.columns = [c.strip().lower() for c in df.columns]
rename = {'timestamp': 'datetime', 'date': 'datetime', 'time': 'datetime'}
for o, n in rename.items():
    if o in df.columns and n not in df.columns:
        df.rename(columns={o: n}, inplace=True)

dt_col = 'datetime' if 'datetime' in df.columns else df.columns[0]
if not pd.api.types.is_datetime64_any_dtype(df[dt_col]):
    df[dt_col] = pd.to_datetime(df[dt_col], errors='coerce')

df = df.sort_values(dt_col).reset_index(drop=True)
df.set_index(dt_col, inplace=True)

for c in ['open', 'high', 'low', 'close']:
    if c not in df.columns:
        raise ValueError(f"Missing column: {c}")
if 'volume' not in df.columns:
    df['volume'] = 1.0

df = df.dropna(subset=['open', 'high', 'low', 'close'])
print(f"  Clean shape: {df.shape}")

# =============================================================================
# FEATURE ENGINEERING
# =============================================================================
print("\n[2/5] Engineering features...")

close = df['close']
high = df['high']
low = df['low']
open_ = df['open']
volume = df['volume']
returns = close.pct_change()

F = pd.DataFrame(index=df.index)

# Basic features
F['ret_1'] = returns
for w in [5, 10, 20, 50]:
    F[f'ret_{w}'] = close.pct_change(w)

for span in [5, 10, 20, 50]:
    ema = close.ewm(span=span, adjust=False).mean()
    F[f'ema_{span}'] = (close - ema) / (ema + 1e-10)

for period in [7, 14, 21]:
    d = close.diff()
    g = d.clip(lower=0)
    l = -d.clip(upper=0)
    ag = g.ewm(alpha=1/period, adjust=False).mean()
    al = l.ewm(alpha=1/period, adjust=False).mean()
    rs = ag / (al + 1e-10)
    F[f'rsi_{period}'] = 100 - 100 / (1 + rs)

macd = close.ewm(span=8, adjust=False).mean() - close.ewm(span=17, adjust=False).mean()
sig = macd.ewm(span=9, adjust=False).mean()
F['macd'] = macd
F['macd_signal'] = sig
F['macd_hist'] = macd - sig

for p in [7, 14, 21]:
    tr = np.maximum(high - low, np.maximum(abs(high - close.shift(1)), abs(low - close.shift(1))))
    F[f'atr_{p}'] = tr.ewm(span=p, adjust=False).mean()
    F[f'atr_ratio_{p}'] = F[f'atr_{p}'] / (close + 1e-10)

for w in [20, 50]:
    ma = close.rolling(w).mean()
    std = close.rolling(w).std()
    F[f'bb_pos_{w}'] = (close - ma) / (std + 1e-10)
    F[f'bb_width_{w}'] = std / (ma + 1e-10)

F['vol_rel_5'] = volume / (volume.rolling(5).mean() + 1e-10)
F['vol_rel_20'] = volume / (volume.rolling(20).mean() + 1e-10)
F['dollar_vol'] = (close * volume) / ((close * volume).rolling(20).mean() + 1e-10)

F['hour'] = df.index.hour / 23.0
F['dow'] = df.index.dayofweek / 6.0
F['month'] = (df.index.month - 1) / 11.0

# Microstructure features
typical = (high + low + close) / 3
vwap = (typical * volume).cumsum() / (volume.cumsum() + 1e-10)
F['vwap_dev'] = (close - vwap) / (vwap + 1e-10)
F['close_loc'] = (close - low) / (high - low + 1e-10)

for w in [10, 20, 50]:
    hh = high.rolling(w).max()
    ll = low.rolling(w).min()
    F[f'price_loc_{w}'] = (close - ll) / (hh - ll + 1e-10)

F['body'] = (close - open_) / (high - low + 1e-10)
F['gap'] = (open_ - close.shift(1)) / close.shift(1)

is_h20 = (close == high.rolling(20).max())
is_l20 = (close == low.rolling(20).min())
F['bars_since_high'] = (~is_h20).iloc[::-1].cumsum().iloc[::-1]
F['bars_since_low'] = (~is_l20).iloc[::-1].cumsum().iloc[::-1]

for w in [10, 30, 60]:
    F[f'realized_vol_{w}'] = returns.rolling(w).std() * np.sqrt(w)

for w in [14, 30]:
    tr = np.maximum(high - low, np.maximum(abs(high - close.shift(1)), abs(low - close.shift(1))))
    atr = tr.rolling(w).mean()
    pdm = (high - high.shift(1)).clip(lower=0)
    mdm = (low.shift(1) - low).clip(lower=0)
    pdi = 100 * pdm.rolling(w).mean() / (atr + 1e-10)
    mdi = 100 * mdm.rolling(w).mean() / (atr + 1e-10)
    dx = 100 * abs(pdi - mdi) / (pdi + mdi + 1e-10)
    F[f'adx_{w}'] = dx.rolling(w).mean()

# Additional interaction features
F['rsi_macd'] = F['rsi_14'] * F['macd_hist']
F['vol_price_loc'] = F['vol_rel_20'] * F['price_loc_20']
F['atr_ret'] = F['atr_14'] * F['ret_1']
F['gap_body'] = F['gap'] * F['body']

print(f"  Total features: {len([c for c in F.columns if not c.startswith('target_')])}")

# =============================================================================
# TARGET GENERATORS
# =============================================================================
def make_targets(horizon):
    ret = close.pct_change(horizon).shift(-horizon)
    targets = {}
    
    # 1. Raw direction
    targets['raw_dir'] = np.sign(ret)
    
    # 2. Vol-filtered direction
    vol = ret.rolling(1000).std()
    targets['vol_dir'] = np.where(abs(ret) > 0.5 * vol, np.sign(ret), 0)
    
    # 3. Momentum-confirmed: only if recent momentum aligns
    mom = close.pct_change(5)
    targets['mom_dir'] = np.where(
        (np.sign(ret) == np.sign(mom)) & (abs(ret) > 0.3 * vol),
        np.sign(ret), 0
    )
    
    # 4. 3-class: strong down / neutral / strong up
    t = ret.rolling(1000).std()
    targets['3class'] = np.where(
        ret > 0.8 * t, 2,
        np.where(ret < -0.8 * t, 0, 1)
    )
    
    # 5. Regression target: signed sqrt return
    targets['reg'] = np.sign(ret) * np.sqrt(abs(ret) + 1e-10)
    
    return ret, targets

# =============================================================================
# EXPERIMENT RUNNER
# =============================================================================
results = []

def run_experiment(name, horizon, target_type, feature_set, lgb_params, use_sample=None):
    """
    feature_set: 'basic' or 'full'
    target_type: 'raw_dir', 'vol_dir', 'mom_dir', '3class', 'reg'
    """
    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {name}")
    print(f"  horizon={horizon}  target={target_type}  features={feature_set}  params={'agg' if lgb_params==LGB_AGGRESSIVE else 'con'}")
    t0 = time.time()
    
    # Select features
    basic_feats = [c for c in F.columns if c in [
        'ret_1','ret_5','ret_10','ret_20','ret_50',
        'ema_5','ema_10','ema_20','ema_50',
        'rsi_7','rsi_14','rsi_21',
        'macd','macd_signal','macd_hist',
        'atr_7','atr_14','atr_21','atr_ratio_7','atr_ratio_14','atr_ratio_21',
        'bb_pos_20','bb_pos_50','bb_width_20','bb_width_50',
        'vol_rel_5','vol_rel_20','dollar_vol',
        'hour','dow','month'
    ]]
    full_feats = [c for c in F.columns if not c.startswith('target_')]
    feat_cols = full_feats if feature_set == 'full' else basic_feats
    
    # Build targets
    ret, targets = make_targets(horizon)
    y_raw = targets[target_type]
    
    # Build working dataframe
    work = F[feat_cols].copy()
    work['__target__'] = y_raw.values
    work['__ret__'] = ret.values
    work = work.dropna()
    
    X = work[feat_cols]
    y = work['__target__'].values
    
    # Handle neutral labels for direction targets
    if target_type in ('raw_dir', 'vol_dir', 'mom_dir'):
        mask = y != 0
        X = X.iloc[mask]
        y = (y[mask] > 0).astype(int)
    
    if len(y) < 10000:
        print(f"  SKIP: too few samples ({len(y)})")
        return None
    
    # Sample for speed if requested
    if use_sample and len(y) > use_sample:
        idx = np.random.choice(len(y), use_sample, replace=False)
        idx.sort()
        X = X.iloc[idx]
        y = y[idx]
    
    # Temporal split
    n = len(X)
    te = int(n * TRAIN_FRAC)
    ve = int(n * (TRAIN_FRAC + VAL_FRAC))
    
    X_train, y_train = X.iloc[:te], y[:te]
    X_val, y_val = X.iloc[te:ve], y[te:ve]
    X_test, y_test = X.iloc[ve:], y[ve:]
    
    print(f"  Samples: train={len(y_train):,} val={len(y_val):,} test={len(y_test):,}")
    
    # Feature selection (MI on train sample)
    if len(X_train) > 200000:
        mi_idx = np.random.choice(len(X_train), 200000, replace=False)
    else:
        mi_idx = np.arange(len(X_train))
    
    try:
        mi = mutual_info_classif(X_train.iloc[mi_idx], y_train[mi_idx], random_state=42, n_neighbors=5)
        mi_df = pd.DataFrame({'f': X_train.columns, 'mi': mi}).sort_values('mi', ascending=False)
        keep = mi_df.head(FEATURE_TOP_K)['f'].tolist()
    except Exception as e:
        print(f"  MI failed: {e}, using all features")
        keep = list(X_train.columns)
    
    X_train = X_train[keep]
    X_val = X_val[keep]
    X_test = X_test[keep]
    
    # Train
    params = dict(lgb_params)
    
    if target_type == 'reg':
        params['objective'] = 'regression'
        params['metric'] = 'rmse'
    elif target_type == '3class':
        params['objective'] = 'multiclass'
        params['metric'] = 'multi_logloss'
        params['num_class'] = 3
    else:
        params['scale_pos_weight'] = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    
    dtrain = lgb.Dataset(X_train, label=y_train)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    
    try:
        model = lgb.train(
            params, dtrain, num_boost_round=MAX_ROUNDS,
            valid_sets=[dtrain, dval], valid_names=['train','valid'],
            callbacks=[
                lgb.early_stopping(EARLY_STOP, verbose=False),
                lgb.log_evaluation(0)
            ]
        )
    except Exception as e:
        print(f"  TRAIN ERROR: {e}")
        return None
    
    best_iter = model.best_iteration
    best_score = model.best_score['valid'].get(params['metric'], 0)
    print(f"  Best iter: {best_iter}  {params['metric']}: {best_score:.5f}")
    
    # Evaluate
    def _eval_split(X_e, y_e, split_name):
        probs = model.predict(X_e, num_iteration=best_iter)
        
        if target_type == 'reg':
            # Convert regression to direction for comparison
            preds_dir = (probs > 0).astype(int)
            true_dir = (y_e > 0).astype(int)
            acc = accuracy_score(true_dir, preds_dir)
            auc = 0.5  # no proper auc for regression
            prec = precision_score(true_dir, preds_dir, zero_division=0)
            rec = recall_score(true_dir, preds_dir, zero_division=0)
            f1 = f1_score(true_dir, preds_dir, zero_division=0)
            ll = log_loss(true_dir, 1/(1+np.exp(-probs/np.std(probs)))) if np.std(probs) > 0 else 1.0
        elif target_type == '3class':
            preds_cls = np.argmax(probs, axis=1)
            acc = accuracy_score(y_e, preds_cls)
            auc = 0.5
            prec = precision_score(y_e, preds_cls, average='macro', zero_division=0)
            rec = recall_score(y_e, preds_cls, average='macro', zero_division=0)
            f1 = f1_score(y_e, preds_cls, average='macro', zero_division=0)
            ll = log_loss(y_e, probs)
        else:
            preds_bin = (probs > 0.5).astype(int)
            acc = accuracy_score(y_e, preds_bin)
            auc = roc_auc_score(y_e, probs)
            prec = precision_score(y_e, preds_bin, zero_division=0)
            rec = recall_score(y_e, preds_bin, zero_division=0)
            f1 = f1_score(y_e, preds_bin, zero_division=0)
            ll = log_loss(y_e, probs)
        
        return {
            f'{split_name}_acc': acc, f'{split_name}_auc': auc,
            f'{split_name}_prec': prec, f'{split_name}_rec': rec,
            f'{split_name}_f1': f1, f'{split_name}_logloss': ll,
        }
    
    metrics = {}
    metrics.update(_eval_split(X_train, y_train, 'train'))
    metrics.update(_eval_split(X_val, y_val, 'val'))
    metrics.update(_eval_split(X_test, y_test, 'test'))
    
    metrics['name'] = name
    metrics['horizon'] = horizon
    metrics['target'] = target_type
    metrics['features'] = feature_set
    metrics['params'] = 'agg' if lgb_params == LGB_AGGRESSIVE else 'con'
    metrics['best_iter'] = best_iter
    metrics['best_score'] = best_score
    metrics['n_features'] = len(keep)
    metrics['time_sec'] = round(time.time() - t0, 1)
    
    print(f"  VAL  -> Acc:{metrics['val_acc']:.4f} AUC:{metrics['val_auc']:.4f} "
          f"Prec:{metrics['val_prec']:.4f} Rec:{metrics['val_rec']:.4f} F1:{metrics['val_f1']:.4f}")
    print(f"  TEST -> Acc:{metrics['test_acc']:.4f} AUC:{metrics['test_auc']:.4f} "
          f"Prec:{metrics['test_prec']:.4f} Rec:{metrics['test_rec']:.4f} F1:{metrics['test_f1']:.4f}")
    
    return {'model': model, 'metrics': metrics, 'features': keep, 'params': params}


# =============================================================================
# RUN EXPERIMENTS
# =============================================================================
print("\n[3/5] Running experiments...")

experiments = [
    # (name, horizon, target_type, feature_set, lgb_params, sample_limit)
    ("E01_baseline_h5_raw_basic_con",    5,  'raw_dir', 'basic', LGB_CONSERVATIVE, 2_000_000),
    ("E02_h20_raw_basic_con",           20,  'raw_dir', 'basic', LGB_CONSERVATIVE, 2_000_000),
    ("E03_h20_vol_basic_con",           20,  'vol_dir', 'basic', LGB_CONSERVATIVE, 2_000_000),
    ("E04_h20_vol_full_con",            20,  'vol_dir', 'full',  LGB_CONSERVATIVE, 2_000_000),
    ("E05_h20_vol_full_agg",            20,  'vol_dir', 'full',  LGB_AGGRESSIVE,   2_000_000),
    ("E06_h60_vol_full_agg",            60,  'vol_dir', 'full',  LGB_AGGRESSIVE,   2_000_000),
    ("E07_h20_mom_full_agg",            20,  'mom_dir', 'full',  LGB_AGGRESSIVE,   2_000_000),
    ("E08_h20_3class_full_agg",         20,  '3class',  'full',  LGB_AGGRESSIVE,   2_000_000),
    ("E09_h20_reg_full_agg",            20,  'reg',     'full',  LGB_AGGRESSIVE,   2_000_000),
]

all_results = []
best_model_data = None
best_val_auc = 0.0

for name, h, tt, fs, lp, samp in experiments:
    try:
        res = run_experiment(name, h, tt, fs, lp, use_sample=samp)
        if res:
            all_results.append(res['metrics'])
            # Track best by val_auc (for classification) or val_acc (for others)
            score = res['metrics']['val_auc'] if res['metrics']['val_auc'] > 0.5 else res['metrics']['val_acc']
            if score > best_val_auc:
                best_val_auc = score
                best_model_data = res
    except Exception as e:
        print(f"  EXPERIMENT FAILED: {e}")
        import traceback
        traceback.print_exc()

# =============================================================================
# SAVE RESULTS
# =============================================================================
print("\n[4/5] Saving results...")

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('val_auc', ascending=False)
results_df.to_csv('experiment_results.csv', index=False)
print(f"  Saved: experiment_results.csv  ({len(results_df)} experiments)")

print("\n" + "=" * 70)
print("LEADERBOARD (by Val AUC)")
print("=" * 70)
for _, row in results_df.head(10).iterrows():
    print(f"  {row['name']:<35}  AUC={row['val_auc']:.4f}  Acc={row['val_acc']:.4f}  "
          f"F1={row['val_f1']:.4f}  Time={row['time_sec']}s")

# Save best model
if best_model_data:
    print("\n[5/5] Saving best model...")
    out = {
        'model': best_model_data['model'],
        'features': best_model_data['features'],
        'params': best_model_data['params'],
        'metrics': best_model_data['metrics'],
    }
    joblib.dump(out, 'best_model.pkl')
    print(f"  Saved: best_model.pkl")
    
    with open('best_config.json', 'w') as f:
        json.dump({
            'name': best_model_data['metrics']['name'],
            'horizon': best_model_data['metrics']['horizon'],
            'target': best_model_data['metrics']['target'],
            'features': best_model_data['metrics']['features'],
            'params': best_model_data['metrics']['params'],
            'val_auc': best_model_data['metrics']['val_auc'],
            'val_acc': best_model_data['metrics']['val_acc'],
            'val_f1': best_model_data['metrics']['val_f1'],
            'n_features': best_model_data['metrics']['n_features'],
            'best_iter': best_model_data['metrics']['best_iter'],
        }, f, indent=2)
    print(f"  Saved: best_config.json")

print("\n" + "=" * 70)
print(f"Done: {datetime.now().isoformat()}")
print("=" * 70)
